# Notebook 08: Dependency & Blast Radius Impact Analysis

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 08 of 12  
**Focus:** Transitive Dependency Traversal, Single Point of Failure (SPOF) Detection, Blast Radius Index, Cycle Auditing  

---

## 1. Business Problem

One of the greatest fears of enterprise IT executives is **unintended decommissioning blast radius**:
- An application is identified as an obsolete, low-utilization candidate for rationalization.
- The project team powers down the server.
- Suddenly, five mission-critical digital banking channels, ATM networks, and fraud prevention engines crash.

Why does this happen? In complex architectures, applications rarely exist in isolation. They communicate via synchronous APIs, ETL batch jobs, shared databases, and message buses. 

Before touching any application, leadership must know its **complete direct and indirect dependency blast radius**.

---

## 2. Core Concept & Formulation

### 2.1 Graph Transitive Traversal (BFS / DFS)
Given target application $A$:
- **Direct Dependents (1-hop):** $\text{Adj}^{-}(A) = \{ u \in V \mid (u, A) \in E_{\text{DEPENDS\_ON}} \}$
- **Indirect Transitive Dependents ($k$-hop):** All nodes $u$ reachable in the reverse graph $G^T$:
  $$\text{Transitive}(A) = \{ u \in V \mid \exists \text{ path from } u \text{ to } A \text{ in } G \}$$

### 2.2 Blast Radius Metric (BRM)
To quantify operational change risk, we define a composite blast radius metric:
$$\text{BRM}(A) = 3 \cdot |\text{Direct}| + 1.5 \cdot |\text{Indirect}| + 4 \cdot |\text{Capabilities}| + 2 \cdot |\text{Impacted BUs}|$$

In [1]:
import sys
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go

workspace_root = Path.cwd().parent
sys.path.insert(0, str(workspace_root / "src"))

from tvi.config import load_config
from tvi.graph import build_knowledge_graph
from tvi.dependency import get_application_dependencies, get_capability_dependencies, get_business_unit_dependencies

config = load_config()
G = build_knowledge_graph()
print(f"Graph Loaded: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Edges")

Graph Loaded: 283 Nodes, 675 Edges


---

## 3. Dependency Cycle Detection & Topological Sort Audit

Before analyzing linear blast radius, we audit the application dependency network for cyclic dependencies (deadlocks / tight coupling):

In [2]:
# Extract App-to-App dependency subgraph
app_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("relationship") == "DEPENDS_ON"]
app_dep_graph = nx.DiGraph(app_edges)

cycles = list(nx.simple_cycles(app_dep_graph))
print(f"=== Dependency Cycle Audit ===")
print(f"Dependency Edges Checked : {len(app_edges)}")
print(f"Cyclic Dependencies Found: {len(cycles)}")
if cycles:
    print(f"Warning: Circular coupling detected: {cycles[:2]}")
else:
    print("✓ Architecture is a Directed Acyclic Graph (DAG) for tested components.")

=== Dependency Cycle Audit ===
Dependency Edges Checked : 32
Cyclic Dependencies Found: 0
✓ Architecture is a Directed Acyclic Graph (DAG) for tested components.


---

## 4. Single Point of Failure (SPOF) Analysis: Scenario D (`APP010`)

We compute the complete direct and transitive blast radius for `APP010` (Central Auth Hub):

In [3]:
app010_dep = get_application_dependencies("APP010", G)

print("=== Blast Radius Dossier: APP010 (Central Auth & Identity Hub) ===")
print(f"Blast Radius Metric (BRM)      : {app010_dep['blast_radius_metric']}")
print(f"Direct Dependent Applications  : {len(app010_dep['direct_dependent_apps'])}")
print(f"Indirect Transitive Dependents : {len(app010_dep['indirect_dependent_apps'])}")
print(f"Capabilities Supported         : {len(app010_dep['capabilities'])}")
print(f"Business Units Impacted        : {len(app010_dep['business_units'])}")

print("\nDirect Dependent Applications List:")
for a in app010_dep["direct_dependent_apps"]:
    print(f"  • {a['id']}: {a['name']}")

print("\nImpacted Business Units:")
for b in app010_dep["business_units"]:
    print(f"  • {b['id']}: {b['name']} ({b['region']})")

=== Blast Radius Dossier: APP010 (Central Auth & Identity Hub) ===
Blast Radius Metric (BRM)      : 40.5
Direct Dependent Applications  : 8
Indirect Transitive Dependents : 3
Capabilities Supported         : 2
Business Units Impacted        : 2

Direct Dependent Applications List:
  • APP001: CoreBanking Alpha
  • APP002: OmniCRM 360
  • APP004: Retail PayFlow Gateway
  • APP005: OrderFlow Enterprise
  • APP011: Customer Direct Portal
  • APP013: TradeExecution Engine
  • APP014: Cloud Payment Microhub
  • APP023: Digital Lending Originator

Impacted Business Units:
  • BU006: Operations & Shared Services (India)
  • BU001: Retail Banking (India)


---

## 5. Bidirectional Blast Radius Comparison: Leaf Node vs Central Hub

We compare the decommissioning risk profile of `APP021` (Leaf Node candidate) vs `APP010` (SPOF Hub):

In [4]:
app021_dep = get_application_dependencies("APP021", G)

comp_df = pd.DataFrame([
    {
        "Application": "APP010 (Central Auth Hub)",
        "Direct Dependent Apps": len(app010_dep["direct_dependent_apps"]),
        "Indirect Dependent Apps": len(app010_dep["indirect_dependent_apps"]),
        "Capabilities Supported": len(app010_dep["capabilities"]),
        "Impacted BUs": len(app010_dep["business_units"]),
        "Blast Radius Score": app010_dep["blast_radius_metric"],
        "Decommissioning Risk": "Extremely High Risk (SPOF Hub)"
    },
    {
        "Application": "APP021 (Legacy Web Portal)",
        "Direct Dependent Apps": len(app021_dep["direct_dependent_apps"]),
        "Indirect Dependent Apps": len(app021_dep["indirect_dependent_apps"]),
        "Capabilities Supported": len(app021_dep["capabilities"]),
        "Impacted BUs": len(app021_dep["business_units"]),
        "Blast Radius Score": app021_dep["blast_radius_metric"],
        "Decommissioning Risk": "Low Risk (Isolated Leaf Node)"
    }
])
display(comp_df)

---

## 6. Demonstrating Capability & Business Unit Dependencies

We demonstrate `get_capability_dependencies()` and `get_business_unit_dependencies()`:

In [5]:
# Capability Dependencies: CAP003 (Order-to-Cash)
cap003_dep = get_capability_dependencies("CAP003", G)
print("=== Capability Dependencies: CAP003 (Order-to-Cash) ===")
print(f"Owning BU: {cap003_dep['owning_business_units']}")
print(f"Supporting Applications: {cap003_dep['supporting_applications']}")
print(f"Targeting Projects: {cap003_dep['targeting_projects']}")

# Business Unit Dependencies: BU001 (Retail Banking)
bu001_dep = get_business_unit_dependencies("BU001", G)
print("\n=== Business Unit Dependencies: BU001 (Retail Banking) ===")
print(f"Owned Capabilities Count: {len(bu001_dep['owned_capabilities'])}")
print(f"Supporting Applications Count: {len(bu001_dep['supporting_applications'])}")

=== Capability Dependencies: CAP003 (Order-to-Cash) ===
Owning BU: [{'id': 'BU004', 'name': 'Digital Payments'}]
Supporting Applications: [{'id': 'APP003', 'name': 'DataMesh Lakehouse', 'support_type': 'Primary'}, {'id': 'APP028', 'name': 'Document Vault Archive', 'support_type': 'Primary'}, {'id': 'APP049', 'name': 'Security Incident Event Mgr', 'support_type': 'Secondary'}, {'id': 'APP005', 'name': 'OrderFlow Enterprise', 'support_type': 'Primary'}, {'id': 'APP012', 'name': 'QuickOrder Legacy', 'support_type': 'Overlapping'}]
Targeting Projects: [{'id': 'PRJ002', 'name': 'NextGen Enterprise Lakehouse', 'status': 'Completed'}, {'id': 'PRJ014', 'name': 'Global Legacy CRM Consolidation', 'status': 'Completed'}, {'id': 'PRJ027', 'name': 'Automated AML Surveillance', 'status': 'Completed'}]

=== Business Unit Dependencies: BU001 (Retail Banking) ===
Owned Capabilities Count: 4
Supporting Applications Count: 12


## 7. Limitations & Analytical Guardrails

1. **Static CMDB vs Dynamic Runtime Logs:** This model reflects documented architectural relationships. In legacy environments, undocumented batch jobs or shared database table dependencies may exist.
2. **Blast Radius Scope:** The metric assesses topological exposure; actual operational risk also depends on data flow volumes and failure tolerance (circuit breakers).

## 8. Next Step

In **Notebook 09: Investment & Benefit Realization**, we connect discretionary technology spending (projects) with business outcomes (benefits and KPIs) in a formal Technology Value Realization (TVR) framework.